# 🏛️ Advanced Named Entity Recognition (NER): From Subword Alignment to Transformer-CRF
### Postgraduate / Ph.D. Level Laboratory & Implementation Guide (Google Colab Ready)
**Author / Lab:** Advanced NLP Research Group  
**Frameworks:** PyTorch 2.x, Hugging Face `transformers`, `seqeval`, `matplotlib`, `seaborn`

---

## 🔬 Theoretical Foundations of Neural Sequence Labeling

### 1. Problem Formulation
Given an untokenized input text sequence $\mathbf{W} = (w_1, w_2, \dots, w_N)$, Named Entity Recognition seeks to assign a sequence of categorical entity tags $\mathbf{y} = (y_1, y_2, \dots, y_N) \in \mathcal{Y}^N$ adhering to the **IOB2 (BIO)** tagging convention:
$$\mathcal{Y} = \{ \text{O}, \text{B-PER}, \text{I-PER}, \text{B-ORG}, \text{I-ORG}, \text{B-LOC}, \text{I-LOC}, \text{B-MISC}, \text{I-MISC} \}$$

### 2. The Subword Tokenization Dilemma
Modern Pretrained Language Models (PLMs) such as BERT, RoBERTa, and DeBERTa operate on subword vocabularies (via WordPiece, BPE, or Unigram SentencePiece). An original word $w_i$ is fractured into subword pieces:
$$w_i \longrightarrow (s_{i, 1}, s_{i, 2}, \dots, s_{i, k_i})$$
This yields a subword sequence $\mathbf{X} = (x_1, x_2, \dots, x_T)$ where $T \ge N$. 
- Naive token classification creates misaligned label spaces.
- **The Postgraduate Solution:** We employ `word_ids()` tracking: the first subword token $s_{i, 1}$ receives the true word label $y_i$, while trailing subwords $s_{i, 2 \dots k_i}$ and special delimiter tokens (`[CLS]`, `[SEP]`, `<s>`, `</s>`) are assigned the canonical PyTorch ignore index:
$$\text{label}(s_{i, j}) = \begin{cases} y_i & \text{if } j = 1 \\ -100 & \text{if } j > 1 \text{ or special token} \end{cases}$$

### 3. Model Architectures: Softmax Independence vs. Linear-Chain CRF
#### (a) Pointwise Independent Softmax (Standard PLM Head):
$$P(\mathbf{y} \mid \mathbf{X}) = \prod_{t=1}^T P(y_t \mid \mathbf{h}_t) = \prod_{t=1}^T \frac{\exp(\mathbf{W}_{y_t}^\top \mathbf{h}_t)}{\sum_{k=1}^{|\mathcal{Y}|} \exp(\mathbf{W}_k^\top \mathbf{h}_t)}$$
*Limitation:* Independence assumption permits illegal transitions (e.g., $P(\text{I-PER} \mid \text{O}) > 0$).

#### (b) Linear-Chain Conditional Random Field (CRF):
A CRF models the global sequence score by combining contextual subword emissions $\mathbf{E} \in \mathbb{R}^{T \times |\mathcal{Y}|}$ with a learnable transition matrix $\mathbf{A} \in \mathbb{R}^{|\mathcal{Y}| \times |\mathcal{Y}|}$:
$$S(\mathbf{X}, \mathbf{y}) = \sum_{t=1}^T \mathbf{E}_{t, y_t} + \sum_{t=1}^{T-1} \mathbf{A}_{y_t, y_{t+1}} + \mathbf{A}_{\text{start}, y_1} + \mathbf{A}_{y_T, \text{end}}$$
The conditional probability of the sequence is normalized over all possible candidate paths $\mathcal{Y}^T$:
$$P(\mathbf{y} \mid \mathbf{X}) = \frac{\exp(S(\mathbf{X}, \mathbf{y}))}{Z(\mathbf{X})}, \quad Z(\mathbf{X}) = \sum_{\mathbf{y}' \in \mathcal{Y}^T} \exp(S(\mathbf{X}, \mathbf{y}'))$$
1. **Partition Function $Z(\mathbf{X})$:** Computed in $\mathcal{O}(T |\mathcal{Y}|^2)$ via the **Forward Algorithm** in logarithmic space using `log-sum-exp`.
2. **Inference (Decoding):** The optimal tag sequence $\mathbf{y}^* = \arg\max_{\mathbf{y}} S(\mathbf{X}, \mathbf{y})$ is solved exactly via the **Viterbi Dynamic Programming Algorithm**.


## 🛠️ Step 1: Environment & GPU Hardware Verification


In [ ]:
# Step 1: Install required research-grade NLP packages
!pip install -q transformers datasets seqeval evaluate accelerate scikit-learn matplotlib seaborn

import os
import sys
import math
import time
import random
import unicodedata
from typing import List, Dict, Tuple, Optional, Any, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    AutoConfig, 
    get_linear_schedule_with_warmup
)
from seqeval.metrics import (
    precision_score, 
    recall_score, 
    f1_score, 
    classification_report
)

# Seed everything for deterministic, reproducible academic experiments
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Execution Compute Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU Hardware: {torch.cuda.get_device_name(0)}")
    print(f"   Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"   PyTorch Version: {torch.__version__}")
    print(f"   Transformers Version: {transformers.__version__}")
else:
    print("   ⚠️ Running on CPU mode. GPU acceleration is recommended for training.")


## 📊 Step 2: Scientific Benchmark Dataset & Label Schema
We establish an academic entity ontology adhering to the standard CoNLL-2003 / OntoNotes standard:
- `PER`: Person entities
- `ORG`: Organizations, corporations, governing bodies
- `LOC`: Geopolitical locations, nations, cities, physical geography
- `MISC`: Miscellaneous named entities, treaties, historical events, cultural terms
- `DATE`: Temporal and calendar entity expressions


In [ ]:
# Step 2: Academic Label Space Definition
ENTITY_LABELS = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC",
    "B-DATE", "I-DATE"
]

LABEL2ID = {label: idx for idx, label in enumerate(ENTITY_LABELS)}
ID2LABEL = {idx: label for idx, label in enumerate(ENTITY_LABELS)}
NUM_LABELS = len(ENTITY_LABELS)

print(f"Ontology Cardinality (|Y|): {NUM_LABELS}")
print("Label to Index Mapping:", LABEL2ID)

# Curated High-Diversity Multilingual and Monolingual Sequence Benchmark Corpus
TRAIN_CORPUS = [
    # Geopolitical and Global Diplomacy
    {
        "tokens": ["United", "Nations", "envoy", "Volodymyr", "Zelenskyy", "met", "with", "Emmanuel", "Macron", "in", "Paris", "on", "15", "October", "."],
        "tags":   ["B-ORG", "I-ORG", "O", "B-PER", "I-PER", "O", "O", "B-PER", "I-PER", "O", "B-LOC", "O", "B-DATE", "I-DATE", "O"]
    },
    {
        "tokens": ["Google", "DeepMind", "researchers", "published", "AlphaFold3", "at", "London", "Headquarters", "in", "2024", "."],
        "tags":   ["B-ORG", "I-ORG", "O", "O", "B-MISC", "O", "B-LOC", "I-LOC", "O", "B-DATE", "O"]
    },
    {
        "tokens": ["The", "Reserve", "Bank", "of", "India", "governor", "Shaktikanta", "Das", "addressed", "parliament", "in", "New", "Delhi", "."],
        "tags":   ["O", "B-ORG", "I-ORG", "I-ORG", "I-ORG", "O", "B-PER", "I-PER", "O", "O", "O", "B-LOC", "I-LOC", "O"]
    },
    {
        "tokens": ["Albert", "Einstein", "formulated", "General", "Relativity", "while", "living", "in", "Berlin", "during", "1915", "."],
        "tags":   ["B-PER", "I-PER", "O", "B-MISC", "I-MISC", "O", "O", "O", "B-LOC", "O", "B-DATE", "O"]
    },
    {
        "tokens": ["Tata", "Consultancy", "Services", "signed", "a", "bilateral", "contract", "with", "Siemens", "in", "Munich", "."],
        "tags":   ["B-ORG", "I-ORG", "I-ORG", "O", "O", "O", "O", "O", "B-ORG", "O", "B-LOC", "O"]
    },
    {
        "tokens": ["Dr.", "B.R.", "Ambedkar", "drafted", "the", "Constitution", "of", "India", "adopted", "in", "November", "1949", "."],
        "tags":   ["O", "B-PER", "I-PER", "O", "O", "B-MISC", "I-MISC", "I-MISC", "O", "O", "B-DATE", "I-DATE", "O"]
    },
    {
        "tokens": ["Sundar", "Pichai", "announced", "new", "Gemini", "multimodal", "architectures", "at", "Mountain", "View", "."],
        "tags":   ["B-PER", "I-PER", "O", "O", "B-MISC", "O", "O", "O", "B-LOC", "I-LOC", "O"]
    },
    {
        "tokens": ["ISRO", "launched", "Chandrayaan-3", "from", "Sriharikota", "spaceport", "on", "14", "July", "."],
        "tags":   ["B-ORG", "O", "B-MISC", "O", "B-LOC", "O", "O", "B-DATE", "I-DATE", "O"]
    },
    {
        "tokens": ["Nelson", "Mandela", "received", "the", "Nobel", "Peace", "Prize", "in", "Oslo", "."],
        "tags":   ["B-PER", "I-PER", "O", "O", "B-MISC", "I-MISC", "I-MISC", "O", "B-LOC", "O"]
    },
    {
        "tokens": ["Microsoft", "and", "OpenAI", "deployed", "supercomputer", "clusters", "across", "California", "."],
        "tags":   ["B-ORG", "O", "B-ORG", "O", "O", "O", "O", "B-LOC", "O"]
    }
]

# Validation / Evaluation Holdout Corpus
VAL_CORPUS = [
    {
        "tokens": ["Satya", "Nadella", "visited", "Microsoft", "Research", "India", "located", "in", "Bengaluru", "last", "Monday", "."],
        "tags":   ["B-PER", "I-PER", "O", "B-ORG", "I-ORG", "I-ORG", "O", "O", "B-LOC", "B-DATE", "I-DATE", "O"]
    },
    {
        "tokens": ["WHO", "officials", "arrived", "in", "Geneva", "for", "the", "World", "Health", "Assembly", "."],
        "tags":   ["B-ORG", "O", "O", "O", "B-LOC", "O", "O", "B-MISC", "I-MISC", "I-MISC", "O"]
    },
    {
        "tokens": ["Rabindranath", "Tagore", "founded", "Visva-Bharati", "University", "near", "Kolkata", "."],
        "tags":   ["B-PER", "I-PER", "O", "B-ORG", "I-ORG", "O", "B-LOC", "O"]
    }
]

print(f"Loaded {len(TRAIN_CORPUS)} training sequences and {len(VAL_CORPUS)} holdout validation sequences.")


## ⚙️ Step 3: Subword Alignment Engine (`word_ids()` & Offset Mapping)
When tokenizing a sentence, Fast Tokenizers fragment arbitrary words into subwords.
- `word_ids()` returns the corresponding index in `tokens` for each subword position.
- We assign `-100` to non-initial subwords and special tokens (`[CLS]`, `[SEP]`, etc.) to instruct PyTorch's loss function to skip them.
- During inference, we invert this mapping using character offsets to extract exact spans from raw text.


In [ ]:
# Step 3: Robust Subword Tokenizer & Alignment
MODEL_CHECKPOINT = "bert-base-cased"  # Can also be "roberta-base", "xlm-roberta-base", "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def align_labels_with_tokens(
    tokens: List[str], 
    tags: List[str], 
    tokenizer: Any, 
    max_length: int = 64
) -> Dict[str, torch.Tensor]:
    """
    Performs canonical subword alignment between word tokens and tokenizer subword IDs.
    Special tokens and non-initial subwords receive index -100.
    """
    tokenized_inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_offsets_mapping=True,
        return_tensors="pt"
    )
    
    word_ids = tokenized_inputs.word_ids(batch_index=0)
    aligned_labels = []
    previous_word_idx = None
    
    for word_idx in word_ids:
        if word_idx is None:
            # Special token ([CLS], [SEP], [PAD])
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            # First subword token corresponding to word_idx
            aligned_labels.append(LABEL2ID[tags[word_idx]])
            previous_word_idx = word_idx
        else:
            # Trailing subword token belonging to same word -> mask with -100
            aligned_labels.append(-100)
            
    input_ids = tokenized_inputs["input_ids"].squeeze(0)
    attention_mask = tokenized_inputs["attention_mask"].squeeze(0)
    label_tensor = torch.tensor(aligned_labels, dtype=torch.long)
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_tensor
    }

# Unit Demonstration on a Fragmented Word: "Shaktikanta"
demo_sample = TRAIN_CORPUS[2]
demo_aligned = align_labels_with_tokens(demo_sample["tokens"], demo_sample["tags"], tokenizer)

print("--- Alignment Verification ---")
tokens_subwords = tokenizer.convert_ids_to_tokens(demo_aligned["input_ids"][:16])
for tok, lab in zip(tokens_subwords, demo_aligned["labels"][:16].tolist()):
    tag_name = ID2LABEL[lab] if lab != -100 else "<IGN (-100)>"
    print(f"Subword: {tok:<15} | Tag: {tag_name}")


## 📦 Step 4: PyTorch Dataset Loader with Dynamic Collation


In [ ]:
# Step 4: PyTorch Dataset Class
class AcademicNERDataset(Dataset):
    """PyTorch Dataset wrapping word tokens, tags, and subword alignment."""
    def __init__(self, corpus: List[Dict[str, Any]], tokenizer: Any, max_length: int = 64):
        self.corpus = corpus
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.encoded_data = [
            align_labels_with_tokens(s["tokens"], s["tags"], tokenizer, max_length)
            for s in corpus
        ]

    def __len__(self) -> int:
        return len(self.encoded_data)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        return self.encoded_data[idx]

train_dataset = AcademicNERDataset(TRAIN_CORPUS, tokenizer)
val_dataset   = AcademicNERDataset(VAL_CORPUS, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False)

print(f"Dataloaders initialized: {len(train_loader)} training batches, {len(val_loader)} validation batches.")


## 🧠 Step 5: Native Linear-Chain Conditional Random Field (CRF)
Here we implement a vectorized, mathematically rigorous **Linear-Chain CRF** in PyTorch:
1. **Forward Algorithm (`forward_alg`):** Evaluates total partition function $Z(\mathbf{X}) = \sum_{\mathbf{y}} \exp(S(\mathbf{X}, \mathbf{y}))$ using numerically stable `log-sum-exp`.
2. **Gold Sequence Score (`score_sentence`):** Computes exact energy $S(\mathbf{X}, \mathbf{y}^\text{gold})$ of the target path.
3. **Negative Log-Likelihood Loss:** $\mathcal{L}_{\text{CRF}} = \log Z(\mathbf{X}) - S(\mathbf{X}, \mathbf{y}^\text{gold})$.
4. **Viterbi Decoding:** Solves for $\mathbf{y}^* = \arg\max_{\mathbf{y}} S(\mathbf{X}, \mathbf{y})$ via dynamic programming backpointers.


In [ ]:
# Step 5: Linear-Chain CRF Implementation
class LinearChainCRF(nn.Module):
    """
    Vectorized Linear-Chain Conditional Random Field (CRF) Layer.
    Guarantees global sequence consistency and prevents illegal IOB2 transitions.
    """
    def __init__(self, num_tags: int, pad_tag_id: int = -100):
        super().__init__()
        self.num_tags = num_tags
        self.pad_tag_id = pad_tag_id

        # Transition parameter matrix: A[i, j] = score of transitioning from tag i to tag j
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags).uniform_(-0.1, 0.1))
        # Start and terminal transitions
        self.start_transitions = nn.Parameter(torch.empty(num_tags).uniform_(-0.1, 0.1))
        self.end_transitions = nn.Parameter(torch.empty(num_tags).uniform_(-0.1, 0.1))

    def forward_alg(self, emissions: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """
        Computes log-partition function Z(X) via the forward recursion in log-space.
        Emissions: (Batch, Seq_Len, Num_Tags)
        Mask: (Batch, Seq_Len) [1 for active, 0 for padded]
        """
        batch_size, seq_len, num_tags = emissions.shape
        # Initialize alpha at t=0
        alpha = self.start_transitions.unsqueeze(0) + emissions[:, 0]  # (B, K)

        for t in range(1, seq_len):
            mask_t = mask[:, t].unsqueeze(1)          # (B, 1)
            emit_t = emissions[:, t].unsqueeze(1)     # (B, 1, K)
            # alpha: (B, K, 1) + trans: (1, K, K) + emit: (B, 1, K) -> (B, K, K)
            next_score = alpha.unsqueeze(2) + self.transitions.unsqueeze(0) + emit_t
            alpha_t = torch.logsumexp(next_score, dim=1)  # Reduce over source tag (dim=1) -> (B, K)
            # Mask update: retain previous score if position is padding
            alpha = torch.where(mask_t.bool(), alpha_t, alpha)

        # Transition to terminal state
        terminal_score = alpha + self.end_transitions.unsqueeze(0)
        return torch.logsumexp(terminal_score, dim=1)  # (B,)

    def score_sentence(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """
        Computes energy S(X, y) of the ground truth tag sequence.
        """
        batch_size, seq_len, _ = emissions.shape
        clean_tags = tags.clone()
        clean_tags[clean_tags == self.pad_tag_id] = 0

        # Score at t = 0
        score = self.start_transitions[clean_tags[:, 0]] + emissions[torch.arange(batch_size), 0, clean_tags[:, 0]]

        for t in range(1, seq_len):
            mask_t = mask[:, t]
            prev_t = clean_tags[:, t - 1]
            curr_t = clean_tags[:, t]

            trans_score = self.transitions[prev_t, curr_t]
            emit_score = emissions[torch.arange(batch_size), t, curr_t]
            score = score + (trans_score + emit_score) * mask_t.float()

        # Terminal state score for last valid non-padded token
        last_valid_indices = torch.clamp(mask.long().sum(dim=1) - 1, min=0)
        last_tags = clean_tags[torch.arange(batch_size), last_valid_indices]
        score = score + self.end_transitions[last_tags]
        return score

    def loss(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """Negative log-likelihood loss: NLL = log Z(X) - S(X, y)"""
        forward_score = self.forward_alg(emissions, mask)
        gold_score = self.score_sentence(emissions, tags, mask)
        return torch.mean(forward_score - gold_score)

    def decode(self, emissions: torch.Tensor, mask: torch.Tensor) -> List[List[int]]:
        """
        Viterbi dynamic programming decoding for globally optimal tag paths.
        """
        batch_size, seq_len, num_tags = emissions.shape
        best_paths = []

        for b in range(batch_size):
            seq_len_b = int(mask[b].sum().item())
            if seq_len_b == 0:
                best_paths.append([])
                continue

            emits_b = emissions[b, :seq_len_b]
            viterbi_vars = self.start_transitions + emits_b[0]
            backpointers = []

            for t in range(1, seq_len_b):
                next_tag_var = viterbi_vars.unsqueeze(1) + self.transitions
                max_vars, bptrs = torch.max(next_tag_var, dim=0)
                viterbi_vars = max_vars + emits_b[t]
                backpointers.append(bptrs.tolist())

            terminal_vars = viterbi_vars + self.end_transitions
            best_tag_id = torch.argmax(terminal_vars).item()

            best_path = [best_tag_id]
            for bptrs_t in reversed(backpointers):
                best_tag_id = bptrs_t[best_tag_id]
                best_path.append(best_tag_id)
            best_path.reverse()
            best_paths.append(best_path)

        return best_paths

print("✅ Vectorized Linear-Chain CRF Module Defined.")


## 🏗️ Step 6: Full Postgraduate Architecture: Transformer-BiLSTM-CRF
This state-of-the-art hybrid architecture integrates:
1. **Pretrained Transformer Backbone (BERT/RoBERTa):** Rich contextualized subword representations.
2. **Bidirectional LSTM Bridge:** Captures directional sequential token recurrence and bridges subword boundaries.
3. **Linear Projection Emission Head:** Projects hidden states into tag emission space $\mathbb{R}^{|\mathcal{Y}|}$.
4. **Linear-Chain CRF Layer:** Globally constrained decoding eliminating invalid transitions.


In [ ]:
# Step 6: Transformer-BiLSTM-CRF Network
class TransformerBiLSTMCRF(nn.Module):
    """
    Hybrid Deep Sequence Labeling Model:
      Transformer Encoder -> BiLSTM Bridge -> Emission Linear Layer -> Linear-Chain CRF
    """
    def __init__(
        self, 
        model_name: str = MODEL_CHECKPOINT, 
        num_tags: int = NUM_LABELS,
        lstm_hidden_dim: int = 256,
        lstm_layers: int = 1,
        dropout_prob: float = 0.2
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        self.hidden_dim = self.config.hidden_size

        # Intermediate BiLSTM sequence bridge
        self.bilstm = nn.LSTM(
            input_size=self.hidden_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            bidirectional=True,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout_prob)
        # 2 * lstm_hidden_dim because bidirectional
        self.emission_head = nn.Linear(lstm_hidden_dim * 2, num_tags)
        self.crf = LinearChainCRF(num_tags=num_tags, pad_tag_id=-100)

    def forward(
        self, 
        input_ids: torch.Tensor, 
        attention_mask: torch.Tensor, 
        labels: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass computing either CRF negative log-likelihood or emission logits.
        """
        # 1. Transformer contextual subword encoding
        transformer_outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        seq_output = transformer_outputs.last_hidden_state  # (Batch, Seq_Len, Hidden)

        # 2. BiLSTM contextualization
        lstm_out, _ = self.bilstm(seq_output)               # (Batch, Seq_Len, 2 * LSTM_Hidden)
        lstm_out = self.dropout(lstm_out)

        # 3. Emission computation
        emissions = self.emission_head(lstm_out)            # (Batch, Seq_Len, Num_Tags)

        # 4. Loss calculation
        loss = None
        if labels is not None:
            # Active mask for valid tokens (ignoring subword positions with -100)
            valid_mask = (labels != -100) & (attention_mask == 1)
            loss = self.crf.loss(emissions, labels, valid_mask)

        return loss, emissions

    def decode(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> List[List[int]]:
        """Viterbi sequence decoding during inference."""
        _, emissions = self.forward(input_ids, attention_mask)
        return self.crf.decode(emissions, attention_mask)

model = TransformerBiLSTMCRF(MODEL_CHECKPOINT, num_tags=NUM_LABELS).to(device)
print(f"✨ Model Instantiated on {device}:")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Total Parameters: {total_params:,} | Trainable: {trainable_params:,}")


## 🚀 Step 7: Optimization with Layer-Wise Learning Rates & Warmup
Postgraduate best practice dictates **differential learning rates**:
- Pretrained Transformer weights require cautious, low learning rates (e.g. $2 \times 10^{-5}$) to prevent catastrophic forgetting.
- Randomly initialized task heads (BiLSTM + CRF transitions) require higher learning rates (e.g. $1 \times 10^{-3}$) for rapid convergence.
- Gradient clipping ($||g||_2 \le 1.0$) stabilizes recurrence and CRF energy optimization.


In [ ]:
# Step 7: Differential Optimizer & Linear Warmup Scheduler
NUM_EPOCHS = 5
GRADIENT_CLIP = 1.0

# Differential parameter grouping
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.transformer.named_parameters()],
        "lr": 2e-5,
        "weight_decay": 0.01
    },
    {
        "params": [p for n, p in model.bilstm.named_parameters()],
        "lr": 1e-3,
        "weight_decay": 0.0
    },
    {
        "params": [p for n, p in model.emission_head.named_parameters()],
        "lr": 1e-3,
        "weight_decay": 0.0
    },
    {
        "params": [p for n, p in model.crf.named_parameters()],
        "lr": 5e-3,
        "weight_decay": 0.0
    }
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

print(f"Optimizer configured with 4 differential parameter groups across {total_steps} total optimization steps.")


## 📈 Step 8: Rigorous Span-Level Evaluation Engine
We evaluate using **Seqeval** (the de-facto academic standard):
- Precision, Recall, and F1 are evaluated on **complete entity spans** (e.g. `[B-PER, I-PER]`), not individual tokens.
- Token accuracy is excluded due to massive class imbalance of `O` tokens.


In [ ]:
# Step 8: Span-Level Evaluation Metric Computation
def evaluate_model(model: nn.Module, data_loader: DataLoader, device: torch.device):
    """
    Computes strict span-level Precision, Recall, and F1 scores using seqeval.
    """
    model.eval()
    all_true_labels = []
    all_pred_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # Viterbi decoding
            decoded_paths = model.decode(input_ids, attention_mask)

            for b in range(len(decoded_paths)):
                gold_seq = labels[b].tolist()
                pred_seq = decoded_paths[b]

                true_tags = []
                pred_tags = []

                for t_idx, gold_tag_id in enumerate(gold_seq):
                    # Only evaluate positions where gold is not -100
                    if gold_tag_id != -100:
                        true_tags.append(ID2LABEL[gold_tag_id])
                        pred_tag_id = pred_seq[t_idx] if t_idx < len(pred_seq) else 0
                        pred_tags.append(ID2LABEL[pred_tag_id])

                if true_tags:
                    all_true_labels.append(true_tags)
                    all_pred_labels.append(pred_tags)

    p = precision_score(all_true_labels, all_pred_labels, zero_division=0)
    r = recall_score(all_true_labels, all_pred_labels, zero_division=0)
    f1 = f1_score(all_true_labels, all_pred_labels, zero_division=0)
    report = classification_report(all_true_labels, all_pred_labels, zero_division=0)

    return {"precision": p, "recall": r, "f1": f1, "report": report}

print("✅ Evaluation Engine with Strict Span Matching Configured.")


## 🏋️ Step 9: Training Execution & Epoch Loss Trajectory


In [ ]:
# Step 9: Model Training Loop with Automatic Mixed Precision (AMP)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
train_loss_history = []

print("=" * 80)
print(f"🚀 INITIATING ADVANCED NER TRAINING ({NUM_EPOCHS} EPOCHS)")
print("=" * 80)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_epoch_loss = 0.0
    start_time = time.time()

    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            loss, _ = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(train_loader)
    train_loss_history.append(avg_loss)
    epoch_duration = time.time() - start_time

    # Periodic Validation
    eval_metrics = evaluate_model(model, val_loader, device)
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS:02d} | Loss: {avg_loss:.4f} | Val F1: {eval_metrics['f1']:.4f} | Time: {epoch_duration:.2f}s")

print("\n" + "=" * 80)
print("FINAL VALIDATION CLASSIFICATION REPORT (SEQEVAL SPAN EVALUATION):")
print("=" * 80)
print(eval_metrics["report"])


## 🎨 Step 10: Academic Diagnostics: CRF Transition Matrix Heatmap
A powerful way to verify whether the CRF has learned true sequence grammar:
- Check that transitioning from `O` to `I-PER` has a severe negative penalty ($A_{O, I-PER} \ll 0$).
- Check that transitioning from `B-PER` to `I-PER` has a strong positive score ($A_{B-PER, I-PER} > 0$).


In [ ]:
# Step 10: Plot Learned Transition Matrix
plt.figure(figsize=(10, 8), dpi=120)
transition_matrix = model.crf.transitions.detach().cpu().numpy()

sns.heatmap(
    transition_matrix,
    annot=True,
    fmt=".2f",
    cmap="Spectral",
    xticklabels=ENTITY_LABELS,
    yticklabels=ENTITY_LABELS,
    cbar_kws={'label': 'Transition Score'}
)
plt.title("Learned Linear-Chain CRF Transition Matrix A[i, j]\n(Rows: Source Tag i, Columns: Destination Tag j)", fontsize=13, fontweight='bold')
plt.xlabel("Destination Tag (t + 1)", fontsize=11)
plt.ylabel("Source Tag (t)", fontsize=11)
plt.tight_layout()
plt.show()


## 🌟 Step 11: Real-Time Inference Pipeline with HTML Entity Highlighter
We reconstruct exact character-level entity spans from raw text and generate an interactive **displaCy-style HTML visualization** inside Colab.


In [ ]:
# Step 11: Production-Grade Inference Pipeline
from IPython.display import HTML, display

def extract_entities_from_text(
    raw_text: str, 
    model: nn.Module, 
    tokenizer: Any, 
    device: torch.device
) -> List[Dict[str, Any]]:
    """
    Inference pipeline mapping subwords back to original character spans in raw text.
    """
    model.eval()
    encoded = tokenizer(
        raw_text,
        return_offsets_mapping=True,
        return_tensors="pt"
    )
    
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)
    offset_mapping = encoded["offset_mapping"][0].tolist()
    
    with torch.no_grad():
        decoded_path = model.decode(input_ids, attention_mask)[0]
        
    extracted_entities = []
    current_entity = None
    
    for idx, (tag_id, (start_char, end_char)) in enumerate(zip(decoded_path, offset_mapping)):
        # Skip special tokens
        if start_char == end_char:
            continue
            
        tag_name = ID2LABEL[tag_id]
        
        if tag_name.startswith("B-"):
            if current_entity:
                extracted_entities.append(current_entity)
            ent_type = tag_name.split("-")[1]
            current_entity = {
                "entity": ent_type,
                "start": start_char,
                "end": end_char,
                "text": raw_text[start_char:end_char]
            }
        elif tag_name.startswith("I-"):
            ent_type = tag_name.split("-")[1]
            if current_entity and current_entity["entity"] == ent_type:
                current_entity["end"] = end_char
                current_entity["text"] = raw_text[current_entity["start"]:end_char]
            else:
                if current_entity:
                    extracted_entities.append(current_entity)
                current_entity = {
                    "entity": ent_type,
                    "start": start_char,
                    "end": end_char,
                    "text": raw_text[start_char:end_char]
                }
        else: # "O"
            if current_entity:
                extracted_entities.append(current_entity)
                current_entity = None
                
    if current_entity:
        extracted_entities.append(current_entity)
        
    return extracted_entities

def render_entities_html(raw_text: str, entities: List[Dict[str, Any]]):
    """Renders rich displaCy-style visual spans in Jupyter / Google Colab."""
    COLORS = {
        "PER": "#a0c4ff",
        "ORG": "#ffd6a5",
        "LOC": "#caffbf",
        "MISC": "#ffadad",
        "DATE": "#fdffb6"
    }
    
    entities = sorted(entities, key=lambda x: x["start"])
    last_idx = 0
    html_parts = []
    
    for ent in entities:
        # Text preceding entity
        html_parts.append(raw_text[last_idx:ent["start"]])
        color = COLORS.get(ent["entity"], "#e0e0e0")
        # Entity badge
        badge = (
            f"<mark style='background-color: {color}; padding: 3px 6px; margin: 0 3px; "
            f"border-radius: 4px; font-weight: 500; font-family: sans-serif; display: inline-block;'>"
            f"{raw_text[ent['start']:ent['end']]} "
            f"<span style='font-size: 0.7em; font-weight: bold; text-transform: uppercase; "
            f"background: rgba(0,0,0,0.15); padding: 2px 4px; border-radius: 3px;'>{ent['entity']}</span>"
            f"</mark>"
        )
        html_parts.append(badge)
        last_idx = ent["end"]
        
    html_parts.append(raw_text[last_idx:])
    full_html = (
        f"<div style='line-height: 2.2; font-size: 1.15rem; font-family: Inter, Roboto, sans-serif; "
        f"padding: 16px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa;'>"
        + "".join(html_parts) +
        "</div>"
    )
    display(HTML(full_html))

# Academic Demonstration Case:
test_query = "Sundar Pichai and Satya Nadella met at the United Nations headquarters in Paris on 15 October."
detected_entities = extract_entities_from_text(test_query, model, tokenizer, device)

print("Extracted Entity Spans (Structured Dictionary):")
for ent in detected_entities:
    print(f"  • {ent['text']:<25} | Type: {ent['entity']:<6} | Span: [{ent['start']}, {ent['end']}]")

print("\nVisual Interactive Span Rendering:")
render_entities_html(test_query, detected_entities)
